# MindSpace AI — Data Science & Keras Training Pipeline
Notebook ini menjadi dokumentasi source code untuk data wrangling, EDA, feature engineering, training TensorFlow/Keras, dan evaluasi model.

In [ ]:
import pandas as pd
from pathlib import Path
ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data/raw/sample_mood_dataset.csv")
df.head()

In [ ]:
question_cols = ["mood", "tidur", "aktivitas", "energi", "stres", "sosial"]
df["stress_reversed"] = 6 - df["stres"]
df["wellbeing_score"] = df["mood"] + df["tidur"] + df["aktivitas"] + df["energi"] + df["stress_reversed"] + df["sosial"]
df["risk_index"] = (6-df["mood"]) + (6-df["tidur"]) + (6-df["energi"]) + df["stres"] + (6-df["sosial"])
df.groupby("label")[["wellbeing_score", "risk_index"]].mean()

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

@tf.keras.utils.register_keras_serializable(package="Custom")
class AttentionLayer(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.attention_weights = self.add_weight(name="attention_weights", shape=(int(input_shape[-1]),), initializer="ones", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return inputs * self.attention_weights

encoder = LabelEncoder()
X = df[question_cols].astype("float32").values
y = encoder.fit_transform(df["label"].astype(str))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
inputs = tf.keras.Input(shape=(6,))
x = AttentionLayer()(inputs)
x = tf.keras.layers.Dense(32, activation="relu")(x)
x = tf.keras.layers.Dense(16, activation="relu")(x)
outputs = tf.keras.layers.Dense(len(encoder.classes_), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

Untuk training lengkap, jalankan script:
```bash
python ml_api/training/train_mood_classifier.py
```